# Aviation Human Factors (HFACS) Baseline Text Classification Pipeline
## B.Tech Research Project Baseline Document

### Project Overview & Research Context
This notebook implements a baseline text classification system to categorize aviation incident and accident narrative reports into the 4 **HFACS Unsafe Act** categories:
1. **Skill-Based Error**: Memory lapses, routine checklist oversights, execution errors during automatic tasks.
2. **Decision Error**: Inappropriate tactical choices, miscalculated risk, misjudged weather/procedures.
3. **Violation**: Willful disregard of rules, SOPs, or regulations.
4. **Perceptual Error**: Visual illusions, spatial disorientation, misjudged visual distances in poor visibility.

### Domain Transfer Test Setup
- **Training Data**: ASRS (Aviation Safety Reporting System) — Self-reported voluntary near-miss reports.
- **Cross-Domain Evaluation Data**: NTSB (National Transportation Safety Board) — Official third-person accident investigation reports.

---

In [ ]:
# Step 0: Import Standard Libraries & Setup Environment
import os
import re
import string
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn NLP & Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# NLTK Preprocessing
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Download NLTK data resources quietly
for resource in ['stopwords', 'punkt', 'punkt_tab']:
    try:
        nltk.data.find(f'tokenizers/{resource}' if 'punkt' in resource else f'corpora/{resource}')
    except LookupError:
        nltk.download(resource, quiet=True)

print("[SETUP COMPLETE] Libraries imported successfully.")

## 1. Modular Function Definitions
Below are the core functions for data loading, preprocessing, baseline training, evaluation, and cross-domain transfer testing.

In [ ]:
# =============================================================================
# Function 1: load_data()
# =============================================================================
def load_data(filepath: str) -> pd.DataFrame:
    """Loads CSV dataset and validates required columns."""
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"File not found: {filepath}")
    df = pd.read_csv(filepath)
    required = {'report_id', 'source', 'text', 'label'}
    if not required.issubset(df.columns):
        raise ValueError(f"Missing required columns: {required - set(df.columns)}")
    print(f"Loaded '{filepath}': {len(df)} records.")
    return df

# =============================================================================
# Function 2: preprocess_text() & preprocess_dataset()
# =============================================================================
def preprocess_text(text: str) -> str:
    """Lowercasing, punctuation stripping, tokenization, stopword removal."""
    if not isinstance(text, str): return ""
    text_clean = text.lower().translate(str.maketrans('', '', string.punctuation))
    try:
        tokens = word_tokenize(text_clean)
    except Exception:
        tokens = text_clean.split()
    try:
        stop_words = set(stopwords.words('english'))
    except Exception:
        stop_words = {'i', 'me', 'my', 'we', 'the', 'a', 'and', 'to', 'for', 'in', 'was'}
    return " ".join([w for w in tokens if w not in stop_words and w.isalpha()])

def preprocess_dataset(df: pd.DataFrame, text_column: str = 'text') -> pd.DataFrame:
    df = df.copy()
    df['clean_text'] = df[text_column].apply(preprocess_text)
    return df

# =============================================================================
# Function 3: train_baseline()
# =============================================================================
def train_baseline(X_train: pd.Series, y_train: pd.Series):
    """Trains TF-IDF + Logistic Regression and LinearSVC baseline models."""
    vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=1, sublinear_tf=True)
    X_train_tfidf = vectorizer.fit_transform(X_train)
    
    models = {
        'Logistic Regression': LogisticRegression(C=1.0, max_iter=1000, random_state=42, class_weight='balanced'),
        'Linear SVM (LinearSVC)': LinearSVC(C=1.0, max_iter=2000, random_state=42, class_weight='balanced', dual='auto')
    }
    for name, model in models.items():
        model.fit(X_train_tfidf, y_train)
        
    return vectorizer, models

# =============================================================================
# Function 4: evaluate_model()
# =============================================================================
def evaluate_model(model, vectorizer, X_test: pd.Series, y_test: pd.Series, 
                   model_name: str, dataset_name: str, labels: list):
    """Evaluates model predictions, prints metrics report, and plots confusion matrix."""
    X_test_tfidf = vectorizer.transform(X_test)
    y_pred = model.predict(X_test_tfidf)
    
    print(f"\n{'='*70}\n EVALUATION: {model_name} | {dataset_name}\n{'='*70}")
    print(f"Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%\n")
    print("Classification Report:")
    print(classification_report(y_test, y_pred, zero_division=0, digits=4))
    
    cm = confusion_matrix(y_test, y_pred, labels=labels)
    plt.figure(figsize=(6, 4.5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
    plt.title(f"Confusion Matrix: {model_name}\n({dataset_name})")
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.show()
    return y_pred, cm

# =============================================================================
# Function 5: transfer_test()
# =============================================================================
def transfer_test(models: dict, vectorizer, ntsb_filepath: str, labels: list):
    """Evaluates trained ASRS models on unseen NTSB accident reports."""
    df_ntsb = load_data(ntsb_filepath)
    df_ntsb = preprocess_dataset(df_ntsb)
    for model_name, model in models.items():
        evaluate_model(model, vectorizer, df_ntsb['clean_text'], df_ntsb['label'],
                       model_name, "Cross-Domain Transfer Test (NTSB)", labels)

## 2. In-Domain Model Training & Evaluation (ASRS Dataset)

In [ ]:
# Load and preprocess training data (ASRS)
df_asrs = load_data('asrs_data.csv')
df_asrs = preprocess_dataset(df_asrs)

# Train/Test Split (80/20 Stratified)
y_counts = df_asrs['label'].value_counts()
stratify_param = df_asrs['label'] if y_counts.min() >= 2 else None

X_train, X_test, y_train, y_test = train_test_split(
    df_asrs['clean_text'], df_asrs['label'], test_size=0.2, random_state=42, stratify=stratify_param
)

# Train Models
vectorizer, models = train_baseline(X_train, y_train)
labels = sorted(list(df_asrs['label'].unique()))

# Evaluate on In-Domain Test Set
for name, model in models.items():
    evaluate_model(model, vectorizer, X_test, y_test, name, "In-Domain Test (ASRS)", labels)

## 3. Cross-Domain Transfer Test (Evaluating on NTSB Dataset)

In [ ]:
# Run Cross-Domain Transfer Test without retraining
transfer_test(models, vectorizer, 'ntsb_data.csv', labels)